# Results Tally (offline)

Optional companion notebook. Point `RESULTS_DIR` at the downloaded `results/`
folder (or attach it as a Kaggle dataset) and this cell produces:

* `summary_results.csv` re-ordered by stage, with the ablation matrix columns,
* `RESULTS_TABLE.md` -- a Markdown table ready to paste into `RESULTS.md`,
* a check that every one of the 13 cells has both an in-domain and an
  out-of-domain number (i.e. no half-measured cells).

It computes nothing new: scores come from the `summary_results.csv` written by
the training sessions.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path("/kaggle/working/results")   # or Path("/kaggle/input/<your-results-dataset>")
RESULTS_DIR = RESULTS_DIR if RESULTS_DIR.exists() else Path("results")

summary_csv = RESULTS_DIR / "metrics" / "summary_results.csv"
if not summary_csv.exists():
    raise SystemExit(f"no summary_results.csv under {RESULTS_DIR}; attach the results folder first")

frame = pd.read_csv(summary_csv)
registry = json.loads((Path("configs") / "experiments_registry.json").read_text())
order = [entry["exp_id"] for entry in registry["experiments"]]
frame["__order"] = frame["exp_id"].map({exp: i for i, exp in enumerate(order)})
frame = frame.sort_values("__order").drop(columns="__order")

display_cols = [
    c for c in [
        "exp_id", "stage", "backbone", "normalization", "augmentation", "status",
        "best_val_macro_f1", "test_id_macro_f1", "test_ood_macro_f1",
        "delta_f1", "rr_f1", "test_id_balanced_acc", "test_ood_balanced_acc",
        "delta_balanced_acc", "rr_balanced_acc",
    ] if c in frame.columns
]
print(frame[display_cols].to_string(index=False))

table = frame[display_cols].to_markdown(index=False)
out_md = RESULTS_DIR / "metrics" / "RESULTS_TABLE.md"
out_md.write_text(table + "\n", encoding="utf-8")
print(f"\nwrote {out_md}")

missing = frame[frame["test_ood_macro_f1"].isna()]["exp_id"].tolist() if "test_ood_macro_f1" in frame else []
print(f"cells without an OOD number: {missing if missing else 'none'}")